# Proof of Concept

Calculate a simple forward pass from 3DGS to 3DGS

In [2]:
from dataset import get_dataloader

loader = get_dataloader("./data/uco3d_data_raw/")
batch = next(iter(loader))
print("Batch loaded!")
print(f"Image shape: {batch['image'].shape}")
print(f"Point cloud shape: {batch['xyz'].shape}")

Batch loaded!
Image shape: torch.Size([4, 3, 512, 512])
Point cloud shape: torch.Size([4, 1024, 3])


In [3]:
import os 
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [ ]:
from uco3d import UCO3DDataset, UCO3DFrameDataBuilder
from uco3d.dataset_utils.utils import get_dataset_root
from uco3d import render_splats

# Get the dataset root folder and check that
# all required metadata files exist.
dataset_root = get_dataset_root(assert_exists=True)
# Get the "small" subset list containing a small subset
# of the uCO3D categories. For loading the whole dataset
# use "set_lists_all-categories.sqlite".
subset_lists_file = os.path.join(
    dataset_root,
    "set_lists", 
    "set_lists_all-categories.sqlite",
)

dataset = UCO3DDataset(
    subset_lists_file=subset_lists_file,
    subsets=["train"],
    frame_data_builder=UCO3DFrameDataBuilder(
        apply_alignment=True,
        load_images=True,
        load_depths=False,
        load_masks=True,
        load_depth_masks=True,
        load_gaussian_splats=True,
        gaussian_splats_truncate_background=True,
        load_point_clouds=True,
        load_segmented_point_clouds=True,
        load_sparse_point_clouds=True,
        box_crop=True,
        box_crop_context=0.4,
        load_frames_from_videos=True,
        image_height=800,
        image_width=800,
        undistort_loaded_blobs=True,
    )
)



torch.Size([44491, 3]) 
 torch.Size([44491, 3]) 
 torch.Size([44491, 1]) 
 torch.Size([44491, 3]) 
 torch.Size([44491, 4]) 



In [ ]:
# query the dataset object to obtain a single video frame of a sequence
frame_data = dataset[100]
# obtain the RGB image of the frame
image_rgb = frame_data.image_rgb
# obtain the 3D gaussian splats reconstructing the whole scene
gaussian_splats = frame_data.sequence_gaussian_splats
# render the scene gaussian splats into the camera of the loaded frame
# NOTE: This requires the 'gsplat' library. You can install it with:
#        > pip install git+https://github.com/nerfstudio-project/gsplat.git@v1.3.0

'''render_colors, render_alphas, render_info = render_splats(
    cameras=frame_data.camera,
    splats=gaussian_splats,
    render_size=[512, 512]
)'''

print(gaussian_splats.means.shape, "\n",
    gaussian_splats.sh0.shape,"\n",
    gaussian_splats.opacities.unsqueeze(1).shape,"\n",
    gaussian_splats.scales.shape,"\n",
    gaussian_splats.quats.shape,"\n" )

In [5]:
import torch

# --- Positions: center + scale to unit sphere ---
means = gaussian_splats.means  # (N, 3)
centroid = means.mean(dim=0, keepdim=True)             # (1, 3)
means_centered = means - centroid
position_scale = means_centered.norm(dim=1).max()      # scalar
means_norm = means_centered / position_scale

# --- Scales: subtract log(position_scale) to stay consistent with position normalization ---
scales_norm = gaussian_splats.scales - torch.log(position_scale)

# --- Quaternions: unit norm + canonical sign (first component non-negative) ---
quats = gaussian_splats.quats
quats_norm = quats / quats.norm(dim=-1, keepdim=True).clamp(min=1e-8)
sign = torch.where(quats_norm[:, :1] < 0, -1.0, 1.0)
quats_norm = quats_norm * sign

# --- sh0 and opacities: z-score normalization ---
sh0 = gaussian_splats.sh0
sh0_mean = sh0.mean(dim=0, keepdim=True)
sh0_std = sh0.std(dim=0, keepdim=True).clamp(min=1e-8)
sh0_norm = (sh0 - sh0_mean) / sh0_std

opacities = gaussian_splats.opacities.unsqueeze(1) if gaussian_splats.opacities.dim() == 1 else gaussian_splats.opacities
op_mean = opacities.mean(dim=0, keepdim=True)
op_std = opacities.std(dim=0, keepdim=True).clamp(min=1e-8)
opacities_norm = (opacities - op_mean) / op_std

# --- Concatenate ---
input = torch.cat([
    means_norm,       # (N, 3)
    sh0_norm,         # (N, 3)
    opacities_norm,   # (N, 1)
    scales_norm,      # (N, 3)
    quats_norm,       # (N, 4)
], dim=-1)            # (N, 14)

# --- Save stats so you can denormalize later ---
norm_stats = {
    "centroid": centroid,
    "position_scale": position_scale,
    "sh0_mean": sh0_mean, "sh0_std": sh0_std,
    "op_mean": op_mean,   "op_std": op_std,
}


In [18]:
input = input.flatten().unsqueeze(0)

input.shape

torch.Size([1, 622874])

In [ ]:
import torch.nn as nn

class POC_Network(nn.Module):
    def __init__(self, dim):
        super(POC_Network, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, dim)
        )

    def forward(self, x):
        x = self.model(x)
        return x

network = POC_Network(dim=input.shape[1])

In [ ]:
from torch.optim import AdamW
import torch.nn.functional as F

optim = AdamW(params=network.parameters(), betas=(0.9, 0.999),eps=1e-8, weight_decay=0.1)

num_epochs = 1000

input.to("mps")

for epoch in range(num_epochs):

    optim.zero_grad()

    recon = network(input)
    loss = F.mse_loss(input, recon)

    loss.backward()
    optim.step()

    if epoch % 50 == 0:
        print(f"step {epoch}: Loss = {loss.item():.6f}")

    

step 0: Loss = 5.985198
step 50: Loss = 4.115030
step 100: Loss = 2.295019
step 150: Loss = 1.021219
step 200: Loss = 0.407251
step 250: Loss = 0.164159
